In [8]:
import sqlite3
from pathlib import Path
import pandas as pd

# Notebook is inside Python_Command
project_root = Path.cwd().parent
raw_folder = project_root / "Datamart" / "Raw"

print("Raw folder:", raw_folder.resolve())

# Find all SQLite databases
database_files = sorted(raw_folder.glob("*.db"))

print(f"Found {len(database_files)} database files:")
for file in database_files:
    print("-", file.name)

Raw folder: C:\Users\User\Desktop\Python 100 Days\Bank_Account_Project\Datamart\Raw
Found 4 database files:
- bbl_2026.db
- kbank_2026.db
- scb_2026.db
- uob_2026.db


In [9]:
all_raw_data = {}
all_bank_frames = []

for database_path in database_files:
    database_name = database_path.stem

    with sqlite3.connect(database_path) as connection:

        # Find tables inside the database
        tables = pd.read_sql_query(
            """
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
            ORDER BY name
            """,
            connection
        )["name"].tolist()

        print(f"\nDatabase: {database_path.name}")
        print("Tables:", tables)

        for table_name in tables:
            df = pd.read_sql_query(
                f'SELECT * FROM "{table_name}"',
                connection
            )

            # Store separately in dictionary
            key = f"{database_name}.{table_name}"
            all_raw_data[key] = df

            # Add database/table source for validation
            df["_source_database"] = database_path.name
            df["_source_table"] = table_name

            all_bank_frames.append(df)

            print(f"Loaded {table_name}: {len(df):,} rows")


Database: bbl_2026.db
Tables: ['bbl_2026']
Loaded bbl_2026: 243 rows

Database: kbank_2026.db
Tables: ['kbank_2026']
Loaded kbank_2026: 9,320 rows

Database: scb_2026.db
Tables: ['scb_2026']
Loaded scb_2026: 10,747 rows

Database: uob_2026.db
Tables: ['uob_2026']
Loaded uob_2026: 181 rows


In [10]:
raw_bank_df = pd.concat(
    all_bank_frames,
    ignore_index=True
)

print("Combined shape:", raw_bank_df.shape)
display(raw_bank_df.head())

Combined shape: (20491, 11)


,bank,account_number,transaction_date,transaction_time,withdrawal,deposit,outstanding_balance,transaction_type,description,_source_database,_source_table
0,BBL,3050XXX982,2026-01-03 00:00:00,03:01,NaN,43435.19,663072.27,Automatic,BBL.CARD,bbl_2026.db,bbl_2026
1,BBL,3050XXX982,2026-01-04 00:00:00,03:00,NaN,110100.82,773173.09,Automatic,BBL.CARD,bbl_2026.db,bbl_2026
2,BBL,3050XXX982,2026-01-05 00:00:00,03:00,NaN,17269.82,790442.91,Automatic,BBL.CARD,bbl_2026.db,bbl_2026
3,BBL,3050XXX982,2026-01-05 00:00:00,04:01,NaN,301.72,790744.63,Automatic,BBL.CARD,bbl_2026.db,bbl_2026
4,BBL,3050XXX982,2026-01-05 00:00:00,09:03,679.45,NaN,789385.73,Automatic,BBL.CARD,bbl_2026.db,bbl_2026


In [11]:
display(
    raw_bank_df.groupby("bank")
    .size()
    .reset_index(name="row_count")
    .sort_values("bank")
)

,bank,row_count
0,BBL,243
1,KBANK,9320
2,SCB,10747
3,UOB,181


In [12]:
# Unique transaction_type values
transaction_type_unique = (
    raw_bank_df["transaction_type"]
    .drop_duplicates()
    .sort_values(na_position="last")
    .reset_index(drop=True)
)

display(transaction_type_unique.to_frame(name="transaction_type"))

,transaction_type
0,LMS SWEEP TRF CR
1,MISC CR
2,MISC CR- IFT
3,MISC DEBIT
4,MISC DR
5,MISC DR ITMX
6,MISC DR-REMIT
7,Automatic
8,Cash Management
9,ค่าธรรมเนียม


In [13]:
import pandas as pd
import re

raw_bank_df = raw_bank_df.copy()

# Clean original transaction type
raw_bank_df["transaction_type_original"] = (
    raw_bank_df["transaction_type"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

def standardize_transaction_type(value):
    if pd.isna(value):
        return "Unknown"

    text = str(value).strip()
    text_upper = text.upper()

    # Opening / carried-forward balance
    if any(keyword in text for keyword in [
        "ยอดยกมา",
        "เปิดบัญชีเป็นเงินสด"
    ]):
        return "Opening Balance"

    # Bank fees and service charges
    if any(keyword in text for keyword in [
        "ค่าธรรมเนียม",
        "ชำระค่าธรรมเนียม",
        "รับดอกเบี้ยเงินฝาก"
    ]):
        if "ดอกเบี้ย" in text:
            return "Interest Income"
        return "Bank Fee"

    # Credit-card payment
    if any(keyword in text for keyword in [
        "ชำระค่าบัตรเครดิต",
        "ชำระบัตรเครดิต"
    ]):
        return "Credit Card Payment"

    # Taxes
    if any(keyword in text for keyword in [
        "ภาษี",
        "หักบัญชีภาษี"
    ]):
        return "Tax Payment"

    # Cheque deposit
    if any(keyword in text for keyword in [
        "ฝากด้วยเช็ค",
        "ฝากเช็ค",
        "เช็ค"
    ]):
        return "Cheque Deposit"

    # Cash deposit
    if any(keyword in text for keyword in [
        "ฝากเงินสด",
        "ฝาก, เงินสด",
        "ฝากถอนเงินสด",
        "ฝาก , เงินสด",
        "ฝากเงินสดผ่านเคาน์เตอร์",
        "ฝาก, ถอนเงินสด-ไม่ใช้สมุด",
        "ฝากถอนเงินโอนไม่ใช้สมุด"
        
    ]):
        return "Cash Deposit"

    # QR receipts
    if any(keyword in text for keyword in [
        "THAI QR PAYMENT",
        "QR PAYMENT"
    ]):
        return "QR Payment Receipt"

    # Sales / merchant receipt
    if any(keyword in text for keyword in [
        "รับเงินจากการขาย",
        "ผ่อนชำระ",
        "คะแนนสะสม"
    ]):
        return "Sales Receipt"

    # Government receipt/payment
    if any(keyword in text for keyword in [
        "ธุรกรรม ตปท.",
        "ธุรกรรมต่างประเทศ"
    ]):
        return "International Transaction"

    # Automatic debit
    if any(keyword in text for keyword in [
        "หักบัญชีอัตโนมัติ",
        "หักบัญชี"
    ]):
        return "Automatic Debit"

    # Sweep transfer
    if "LMS SWEEP" in text_upper:
        return "Sweep Transfer"

    # Cash management
    if "CASH MANAGEMENT" in text_upper:
        return "Cash Management"

    # Automatic transaction
    if "AUTOMATIC" in text_upper:
        return "Automatic Transaction"

    # Miscellaneous credit
    if any(keyword in text_upper for keyword in [
        "MISC CR",
        "MISC CREDIT"
    ]):
        return "Other Credit"

    # Miscellaneous debit
    if any(keyword in text_upper for keyword in [
        "MISC DR",
        "MISC DEBIT"
    ]):
        return "Other Debit"

    # Incoming transfer
    if any(keyword in text for keyword in [
        "รับโอนเงิน",
        "รับโอนเงินผ่านเขต",
        "รับโอนเงินอัตโนมัติ"
    ]):
        return "Incoming Transfer"

    # Outgoing transfer
    if any(keyword in text for keyword in [
        "โอนเงิน",
        "โอนบัญชี",
        "โอนบัญชีเป็นเงินสด"
    ]):
        return "Outgoing Transfer"

    return "Other"


raw_bank_df["transaction_type_standard"] = (
    raw_bank_df["transaction_type_original"]
    .apply(standardize_transaction_type)
)

In [14]:
transaction_type_mapping = (
    raw_bank_df[
        [
            "transaction_type_original",
            "transaction_type_standard"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "transaction_type_standard",
            "transaction_type_original"
        ]
    )
    .reset_index(drop=True)
)

display(transaction_type_mapping)

,transaction_type_original,transaction_type_standard
0,หักบัญชี,Automatic Debit
1,หักบัญชีอัตโนมัติ,Automatic Debit
2,Automatic,Automatic Transaction
3,ค่าธรรมเนียม,Bank Fee
4,ค่าธรรมเนียม SMS ขยันบอก / อื่น ๆ,Bank Fee
5,ชำระค่าธรรมเนียมการใช้เครื่อง EDC,Bank Fee
6,"ฝาก , เงินสด",Cash Deposit
7,"ฝาก, ถอนเงินสด-ไม่ใช้สมุด",Cash Deposit
8,ฝากถอนเงินโอนไม่ใช้สมุด,Cash Deposit
9,ฝากเงินสด,Cash Deposit


In [15]:
# Keep the original description unchanged
raw_bank_df["description_original"] = raw_bank_df["description"]

# Basic description cleaning
raw_bank_df["description_clean"] = (
    raw_bank_df["description"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

In [16]:
NO_CUSTOMER_TYPES = {
    "Bank Fee",
    "Interest Income",
    "Opening Balance",
    "Tax Payment",
    "Cash Deposit",
    "Cheque Deposit",
    "Credit Card Payment",
    "Sweep Transfer",
    "Automatic Transaction",
}


def clean_customer_text(text):
    """Remove common bank and transaction wording from a possible name."""

    if pd.isna(text):
        return pd.NA

    text = str(text).strip()

    # Bank names and abbreviations
    text = re.sub(
        r"\b(?:KBANK|KASIKORN|SCB|BBL|BANGKOK BANK|UOB|KTB|TTB|BAY|KRUNGSRI)\b",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Masked and normal account references
    text = re.sub(
        r"\b(?:X+|\*+)\d{3,}\b",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\b\d{6,16}\b",
        " ",
        text
    )

    # Common transaction words
    phrases = [
        r"รับโอนเงินจาก",
        r"รับโอนจาก",
        r"รับเงินจาก",
        r"โอนเงินให้",
        r"โอนไป",
        r"โอนจาก",
        r"โอนเงิน",
        r"ผ่านเขต",
        r"ข้ามเขต",
        r"อัตโนมัติ",
        r"CROSSBANK",
        r"TRANSFER",
        r"REMIT",
        r"REMITTANCE",
        r"PAYMENT",
        r"RECEIVED",
        r"ON GOODS",
        r"MISC CR",
        r"MISC DR",
        r"MISC CREDIT",
        r"MISC DEBIT",
        r"CARD SETTLEMENT",
        r"CREDIT CARD DIVISION",
        r"THAI QR PAYMENT",
        r"QR PAYMENT",
        r"BBL\.CARD",
        r"EDC",
    ]

    for phrase in phrases:
        text = re.sub(
            phrase,
            " ",
            text,
            flags=re.IGNORECASE
        )

    # Remove separators and noise
    text = re.sub(r"[|,/\\:_+=]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"^[\-.]+|[\-+.*]+$", "", text).strip()

    if not text:
        return pd.NA

    # Reject results without Thai or English letters
    if not re.search(r"[A-Za-zก-๙]", text):
        return pd.NA

    return text

In [17]:
def extract_customer_by_transaction_type(row):
    transaction_type = row.get("transaction_type_standard")
    description = row.get("description_clean")

    if pd.isna(description):
        return pd.NA

    text = str(description).strip()

    # Categories where description normally does not represent a customer
    if transaction_type in NO_CUSTOMER_TYPES:
        return pd.NA

    # Incoming transfer: extract sender
    if transaction_type == "Incoming Transfer":
        patterns = [
            r"รับโอนเงินจาก\s+(.+)",
            r"รับโอนจาก\s+(.+)",
            r"รับเงินจาก\s+(.+)",
            r"FROM\s+(.+)",
        ]

        for pattern in patterns:
            match = re.search(pattern, text, flags=re.IGNORECASE)

            if match:
                return clean_customer_text(match.group(1))

        return clean_customer_text(text)

    # Outgoing transfer: extract recipient
    if transaction_type == "Outgoing Transfer":
        patterns = [
            r"โอนเงินให้\s+(.+)",
            r"โอนไป\s+(.+)",
            r"โอนเงิน\s+(.+)",
            r"TO\s+(.+)",
        ]

        for pattern in patterns:
            match = re.search(pattern, text, flags=re.IGNORECASE)

            if match:
                return clean_customer_text(match.group(1))

        return clean_customer_text(text)

    # Sales receipt: extract payer only when description contains useful text
    if transaction_type == "Sales Receipt":
        # Generic settlement descriptions should not become customer names
        generic_sales_descriptions = [
            r"^BBL\.CARD$",
            r"^UOB CARD SETTLEMENT$",
            r"^CREDIT CARD DIVISION",
            r"^THAI QR PAYMENT$",
            r"^EDC$",
        ]

        for pattern in generic_sales_descriptions:
            if re.search(pattern, text, flags=re.IGNORECASE):
                return pd.NA

        return clean_customer_text(text)

    # Other credit can represent unidentified incoming money
    if transaction_type == "Other Credit":
        return clean_customer_text(text)

    # Other debit can represent vendor, employee or another recipient
    if transaction_type == "Other Debit":
        return clean_customer_text(text)

    # Cash management may include counterparties, but use lower confidence
    if transaction_type == "Cash Management":
        return clean_customer_text(text)

    # International transactions may contain a foreign counterparty
    if transaction_type == "International Transaction":
        return clean_customer_text(text)

    # Automatic debit may contain a vendor or service provider
    if transaction_type == "Automatic Debit":
        return clean_customer_text(text)

    return pd.NA

In [18]:
raw_bank_df["customer_name_raw"] = raw_bank_df.apply(
    extract_customer_by_transaction_type,
    axis=1
)

In [19]:
raw_bank_df["customer_name"] = (
    raw_bank_df["customer_name_raw"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.replace(r"[+*]+$", "", regex=True)
    .str.upper()
)

In [20]:
customer_role_map = {
    "Incoming Transfer": "Sender",
    "Outgoing Transfer": "Recipient",
    "Sales Receipt": "Customer/Payer",
    "Other Credit": "Possible Sender",
    "Other Debit": "Possible Recipient",
    "Automatic Debit": "Possible Payee",
    "International Transaction": "Counterparty",
    "Cash Management": "Possible Counterparty",
}

raw_bank_df["counterparty_role"] = (
    raw_bank_df["transaction_type_standard"]
    .map(customer_role_map)
)

In [21]:
raw_bank_df["customer_name_confidence"] = "Not Extracted"

high_confidence_mask = (
    raw_bank_df["customer_name"].notna()
    & raw_bank_df["transaction_type_standard"].isin(
        ["Incoming Transfer", "Outgoing Transfer"]
    )
    & raw_bank_df["description_clean"].str.contains(
        r"รับโอนจาก|รับโอนเงินจาก|โอนเงินให้|โอนไป|FROM|TO",
        case=False,
        na=False,
        regex=True
    )
)

medium_confidence_mask = (
    raw_bank_df["customer_name"].notna()
    & raw_bank_df["transaction_type_standard"].isin(
        [
            "Incoming Transfer",
            "Outgoing Transfer",
            "Sales Receipt",
            "International Transaction",
        ]
    )
)

low_confidence_mask = (
    raw_bank_df["customer_name"].notna()
    & raw_bank_df["transaction_type_standard"].isin(
        [
            "Other Credit",
            "Other Debit",
            "Automatic Debit",
            "Cash Management",
        ]
    )
)

raw_bank_df.loc[
    low_confidence_mask,
    "customer_name_confidence"
] = "Low"

raw_bank_df.loc[
    medium_confidence_mask,
    "customer_name_confidence"
] = "Medium"

raw_bank_df.loc[
    high_confidence_mask,
    "customer_name_confidence"
] = "High"

In [22]:
display(
    raw_bank_df[
        [
            "bank",
            "transaction_type_original",
            "transaction_type_standard",
            "description_original",
            "customer_name",
            "counterparty_role",
            "customer_name_confidence",
        ]
    ]
    .drop_duplicates()
    .head(200)
)

,bank,transaction_type_original,transaction_type_standard,description_original,customer_name,counterparty_role,customer_name_confidence
0,BBL,Automatic,Automatic Transaction,BBL.CARD,<NA>,NaN,Not Extracted
18,BBL,Cash Management,Cash Management,Vendor Payment,VENDOR,Possible Counterparty,Low
43,BBL,Cash Management,Cash Management,RECEIVED ON GOODS,<NA>,Possible Counterparty,Not Extracted
54,BBL,Automatic,Automatic Transaction,Fees 26011223011901825058,<NA>,NaN,Not Extracted
62,BBL,Automatic,Automatic Transaction,Fees 26011923001996276024,<NA>,NaN,Not Extracted
...,...,...,...,...,...,...,...
468,KBANK,โอนเงิน,Outgoing Transfer,โอนไป SCB X3569 นางสาว บุณยาพร มีแ++,นางสาว บุณยาพร มีแ,Recipient,High
469,KBANK,โอนเงิน,Outgoing Transfer,โอนไป X7823 บจก. วาย.เอส.เอส.พ++,บจก. วาย.เอส.เอส.พ,Recipient,High
470,KBANK,โอนเงิน,Outgoing Transfer,โอนไป BBL X2654 นาย ศุภกฤต แย้มวา++,นาย ศุภกฤต แย้มวา,Recipient,High
471,KBANK,โอนเงิน,Outgoing Transfer,โอนไป SCB X7641 นางสาว ทัศนีย์ คำพ++,นางสาว ทัศนีย์ คำพ,Recipient,High


In [23]:
extraction_summary = (
    raw_bank_df
    .groupby("transaction_type_standard", dropna=False)
    .agg(
        total_rows=("description", "size"),
        customer_names_extracted=(
            "customer_name",
            lambda x: x.notna().sum()
        )
    )
    .reset_index()
)

extraction_summary["extraction_rate"] = (
    extraction_summary["customer_names_extracted"]
    / extraction_summary["total_rows"]
)

display(
    extraction_summary.sort_values(
        "total_rows",
        ascending=False
    )
)

,transaction_type_standard,total_rows,customer_names_extracted,extraction_rate
3,Cash Deposit,10714,0,0.0
7,Incoming Transfer,6403,6403,1.0
13,Outgoing Transfer,1720,1720,1.0
14,Sales Receipt,961,961,1.0
4,Cash Management,172,24,0.139535
11,Other Credit,167,0,0.0
2,Bank Fee,81,0,0.0
1,Automatic Transaction,71,0,0.0
5,Cheque Deposit,40,0,0.0
16,Tax Payment,39,0,0.0


In [24]:
# 1. Show all column names
print("Columns:")
print(raw_bank_df.columns.tolist())

Columns:
['bank', 'account_number', 'transaction_date', 'transaction_time', 'withdrawal', 'deposit', 'outstanding_balance', 'transaction_type', 'description', '_source_database', '_source_table', 'transaction_type_original', 'transaction_type_standard', 'description_original', 'description_clean', 'customer_name_raw', 'customer_name', 'counterparty_role', 'customer_name_confidence']


In [25]:
# 2. Check data types
print("\nData types:")
print(raw_bank_df.dtypes)


Data types:
bank                             str
account_number                   str
transaction_date                 str
transaction_time                 str
withdrawal                   float64
deposit                      float64
outstanding_balance          float64
transaction_type                 str
description                      str
_source_database                 str
_source_table                    str
transaction_type_original     string
transaction_type_standard        str
description_original             str
description_clean             string
customer_name_raw                str
customer_name                 string
counterparty_role                str
customer_name_confidence         str
dtype: object


In [26]:
# 3. Check shape
print("\nShape:")
print(raw_bank_df.shape)


Shape:
(20491, 19)


In [27]:
# 4. Check missing values
column_check = pd.DataFrame({
    "column": raw_bank_df.columns,
    "dtype": raw_bank_df.dtypes.astype(str).values,
    "non_null_count": raw_bank_df.notna().sum().values,
    "null_count": raw_bank_df.isna().sum().values,
    "null_percentage": (
        raw_bank_df.isna().mean().mul(100).round(2).values
    ),
    "unique_count": raw_bank_df.nunique(dropna=True).values
})

display(column_check)

,column,dtype,non_null_count,null_count,null_percentage,unique_count
0,bank,str,20491,0,0.00,4
1,account_number,str,20491,0,0.00,25
2,transaction_date,str,20491,0,0.00,187
3,transaction_time,str,13075,7416,36.19,1024
4,withdrawal,float64,2398,18093,88.30,1162
5,deposit,float64,18069,2422,11.82,8769
6,outstanding_balance,float64,20491,0,0.00,20434
7,transaction_type,str,20491,0,0.00,35
8,description,str,20468,23,0.11,11059
9,_source_database,str,20491,0,0.00,4


In [28]:
import sqlite3
from pathlib import Path
import pandas as pd

# Create transaction type dimension
dim_transaction_type = (
    raw_bank_df[["transaction_type_standard"]]
    .dropna()
    .drop_duplicates()
    .sort_values("transaction_type_standard")
    .reset_index(drop=True)
)

# Create surrogate key
dim_transaction_type.insert(
    0,
    "transaction_type_key",
    range(1, len(dim_transaction_type) + 1)
)

# Optional renamed display column
dim_transaction_type = dim_transaction_type.rename(
    columns={
        "transaction_type_standard": "transaction_type_name"
    }
)

display(dim_transaction_type)
print(dim_transaction_type.shape)

,transaction_type_key,transaction_type_name
0,1,Automatic Debit
1,2,Automatic Transaction
2,3,Bank Fee
3,4,Cash Deposit
4,5,Cash Management
5,6,Cheque Deposit
6,7,Credit Card Payment
7,8,Incoming Transfer
8,9,Interest Income
9,10,International Transaction


(17, 2)


In [29]:
# Detect project root
current_folder = Path.cwd()

if current_folder.name.lower() == "python_command":
    project_root = current_folder.parent
else:
    project_root = current_folder

# Datamart/Dim destination
dim_folder = project_root / "Datamart" / "Dim"
dim_folder.mkdir(parents=True, exist_ok=True)

database_path = dim_folder / "dim_database.db"
table_name = "dim_transaction_type"

with sqlite3.connect(database_path) as connection:
    dim_transaction_type.to_sql(
        name=table_name,
        con=connection,
        if_exists="replace",
        index=False
    )

print(f"Database: {database_path.resolve()}")
print(f"Table: {table_name}")
print(f"Rows exported: {len(dim_transaction_type):,}")

Database: C:\Users\User\Desktop\Python 100 Days\Bank_Account_Project\Datamart\Dim\dim_database.db
Table: dim_transaction_type
Rows exported: 17


In [30]:
dim_counterparty_role = (
    raw_bank_df[["counterparty_role"]]
    .dropna()
    .drop_duplicates()
    .sort_values("counterparty_role")
    .reset_index(drop=True)
)

dim_counterparty_role.insert(
    0,
    "counterparty_role_key",
    range(1, len(dim_counterparty_role) + 1)
)

dim_counterparty_role = dim_counterparty_role.rename(
    columns={
        "counterparty_role": "counterparty_role_name"
    }
)

display(dim_counterparty_role)

,counterparty_role_key,counterparty_role_name
0,1,Counterparty
1,2,Customer/Payer
2,3,Possible Counterparty
3,4,Possible Payee
4,5,Possible Recipient
5,6,Possible Sender
6,7,Recipient
7,8,Sender


In [31]:
dim_customer_name_confidence = (
    raw_bank_df[["customer_name_confidence"]]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

# Optional logical ordering
confidence_order = {
    "High": 1,
    "Medium": 2,
    "Low": 3,
    "Not Extracted": 4
}

dim_customer_name_confidence["sort_order"] = (
    dim_customer_name_confidence["customer_name_confidence"]
    .map(confidence_order)
    .fillna(99)
    .astype(int)
)

dim_customer_name_confidence = (
    dim_customer_name_confidence
    .sort_values(
        ["sort_order", "customer_name_confidence"]
    )
    .reset_index(drop=True)
)

dim_customer_name_confidence.insert(
    0,
    "customer_name_confidence_key",
    range(1, len(dim_customer_name_confidence) + 1)
)

dim_customer_name_confidence = (
    dim_customer_name_confidence.rename(
        columns={
            "customer_name_confidence":
                "customer_name_confidence_name"
        }
    )
)

display(dim_customer_name_confidence)

,customer_name_confidence_key,customer_name_confidence_name,sort_order
0,1,High,1
1,2,Medium,2
2,3,Low,3
3,4,Not Extracted,4


In [32]:
current_folder = Path.cwd()

if current_folder.name.lower() == "python_command":
    project_root = current_folder.parent
else:
    project_root = current_folder

dim_folder = project_root / "Datamart" / "Dim"
dim_folder.mkdir(parents=True, exist_ok=True)

database_path = dim_folder / "dim_database.db"

with sqlite3.connect(database_path) as connection:

    dim_counterparty_role.to_sql(
        name="dim_counterparty_role",
        con=connection,
        if_exists="replace",
        index=False
    )

    dim_customer_name_confidence.to_sql(
        name="dim_customer_name_confidence",
        con=connection,
        if_exists="replace",
        index=False
    )

print(f"Database: {database_path.resolve()}")
print(
    f"dim_counterparty_role: "
    f"{len(dim_counterparty_role):,} rows"
)
print(
    f"dim_customer_name_confidence: "
    f"{len(dim_customer_name_confidence):,} rows"
)

Database: C:\Users\User\Desktop\Python 100 Days\Bank_Account_Project\Datamart\Dim\dim_database.db
dim_counterparty_role: 8 rows
dim_customer_name_confidence: 4 rows


In [33]:
# Work on a copy
staging_bank_df = raw_bank_df.copy()

# Columns to keep
staging_columns = [
    "bank",
    "account_number",
    "transaction_date",
    "transaction_time",
    "withdrawal",
    "deposit",
    "outstanding_balance",
    "transaction_type",
    "transaction_type_standard",
    "description_clean",
    "customer_name",
    "counterparty_role",
    "customer_name_confidence"
]

# Keep only columns that currently exist
existing_columns = [
    column
    for column in staging_columns
    if column in staging_bank_df.columns
]

missing_columns = [
    column
    for column in staging_columns
    if column not in staging_bank_df.columns
]

if missing_columns:
    print("Missing columns:", missing_columns)

staging_bank_df = staging_bank_df[existing_columns].copy()

In [34]:
# Transaction date
staging_bank_df["transaction_date"] = pd.to_datetime(
    staging_bank_df["transaction_date"],
    errors="coerce"
)

# Standard text columns
text_columns = [
    "bank",
    "account_number",
    "transaction_time",
    "transaction_type",
    "transaction_type_standard",
    "description_clean",
    "customer_name",
    "counterparty_role",
    "customer_name_confidence"
]

for column in text_columns:
    if column in staging_bank_df.columns:
        staging_bank_df[column] = (
            staging_bank_df[column]
            .astype("string")
            .str.strip()
        )

# Numeric columns
numeric_columns = [
    "withdrawal",
    "deposit",
    "outstanding_balance"
]

for column in numeric_columns:
    if column in staging_bank_df.columns:
        staging_bank_df[column] = pd.to_numeric(
            staging_bank_df[column],
            errors="coerce"
        )

# Remove rows without a valid transaction date
staging_bank_df = staging_bank_df.dropna(
    subset=["transaction_date"]
).copy()

In [35]:
# Detect the project root
current_folder = Path.cwd()

if current_folder.name.lower() == "python_command":
    project_root = current_folder.parent
else:
    project_root = current_folder

# Create the staging folder
staging_folder = project_root / "Datamart" / "Staging"
staging_folder.mkdir(parents=True, exist_ok=True)

# Split and export by transaction year
for year, year_df in staging_bank_df.groupby(
    staging_bank_df["transaction_date"].dt.year
):
    year = int(year)

    database_path = (
        staging_folder
        / f"staging_bank_statement_{year}.db"
    )

    table_name = f"staging_bank_statement_{year}"

    year_df = year_df.copy()

    # SQLite-compatible datetime string
    year_df["transaction_date"] = (
        year_df["transaction_date"]
        .dt.strftime("%Y-%m-%d")
    )

    with sqlite3.connect(database_path) as connection:
        year_df.to_sql(
            name=table_name,
            con=connection,
            if_exists="replace",
            index=False,
            chunksize=10_000
        )

    print(f"\nYear: {year}")
    print(f"Rows exported: {len(year_df):,}")
    print(f"Database: {database_path.resolve()}")
    print(f"Table: {table_name}")


Year: 2026
Rows exported: 20,491
Database: C:\Users\User\Desktop\Python 100 Days\Bank_Account_Project\Datamart\Staging\staging_bank_statement_2026.db
Table: staging_bank_statement_2026
